# LeetCode 1072 — Flip Columns For Maximum Number of Equal Rows bằng Python

Link đề: https://leetcode.com/problems/flip-columns-for-maximum-number-of-equal-rows/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Matrix

Skeleton LeetCode:

```python
class Solution:
    def maxEqualRowsAfterFlips(self, matrix: List[List[int]]) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất**: số hàng nhiều nhất có thể có **mọi phần tử bằng nhau** (toàn `0` hoặc toàn `1`) sau khi lật một số cột tuỳ ý. Điểm cốt lõi: lật cột tác động **cùng một lúc lên mọi hàng**, nên hai hàng cùng trở nên "đều" ⇔ chúng **giống nhau hoặc là bù của nhau** (hoàn toàn đối lập). Chuẩn hoá mỗi hàng (lật cả hàng nếu phần tử đầu là `1`) rồi **đếm bằng bảng băm**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: ma trận nhị phân `m × n`; thao tác lật **cả một cột**; tối đa hoá số hàng mà mọi ô trong hàng bằng nhau.
2. Nắm công cụ: bù (complement) của hàng nhị phân, chuẩn hoá bằng XOR với phần tử đầu, `Counter`/`dict` của `tuple`, biểu diễn hàng bằng số nguyên bit.
3. Đi từ **vét cạn `2^n` tập cột lật** → **so sánh từng cặp hàng** → **chuẩn hoá + đếm** → **khoá `min(hàng, bù)`** → **số nguyên bit**.
4. Hiểu mấu chốt: một tập cột lật `F` biến hàng `r` thành `r XOR F`; hàng này đều ⇔ `r = F` hoặc `r = bù(F)`. Vậy các hàng cùng đều sau cùng một `F` đúng là các hàng thuộc **lớp {r, bù(r)}**; đáp án là kích thước lớp lớn nhất.
5. Tránh lỗi kinh điển: chỉ đếm hàng giống hệt (bỏ bù), nhóm theo số lượng số `1`, chỉ đếm hàng đã đều sẵn, nhầm "bù" với "đảo ngược", lật hàng thay vì lật cột.

## 2. Đề bài & ví dụ

Cho ma trận nhị phân `matrix` cỡ `m × n`. Bạn có thể chọn **bất kỳ số cột nào** và **lật mọi ô** của các cột đó (`0` thành `1`, `1` thành `0`).

Trả về **số hàng nhiều nhất** mà tất cả giá trị trong hàng bằng nhau sau một số thao tác lật nào đó.

**Ví dụ 1:** `matrix = [[0,1],[1,1]]` → `1` (không lật gì, một hàng `[1,1]` đều).

**Ví dụ 2:** `matrix = [[0,1],[1,0]]` → `2` (lật cột đầu, cả hai hàng thành `[1,1]` và `[0,0]`).

**Ví dụ 3:** `matrix = [[0,0,0],[0,0,1],[1,1,0]]` → `2` (lật hai cột đầu: hai hàng cuối thành `[1,1,1]` và `[0,0,0]`; hoặc lật cột cuối: hàng 2 và 3 thành `[0,0,0]` và `[1,1,1]`).

**Ràng buộc:** `1 <= m, n <= 300`; `matrix[i][j]` là `0` hoặc `1`.

In [1]:
import itertools
import random
import time
from collections import Counter
from typing import List

examples = [
    ([[0, 1], [1, 1]], 1),
    ([[0, 1], [1, 0]], 2),
    ([[0, 0, 0], [0, 0, 1], [1, 1, 0]], 2),
]
for mat, want in examples:
    print(f"matrix = {mat} -> {want}")

matrix = [[0, 1], [1, 1]] -> 1
matrix = [[0, 1], [1, 0]] -> 2
matrix = [[0, 0, 0], [0, 0, 1], [1, 1, 0]] -> 2


## 3. Kiến thức nền

### 3.1 Lật cột là XOR cùng một mặt nạ

Chọn tập cột lật `F ∈ {0,1}^n` (`F[j] = 1` nếu lật cột `j`). Ô `(i, j)` trở thành `matrix[i][j] XOR F[j]`. Vậy hàng `r` trở thành `r XOR F` — **mọi hàng bị XOR cùng một `F`**.

### 3.2 Hàng đều và bù

Hàng `r XOR F` là hàng đều (toàn `0` hoặc toàn `1`) ⇔ `r XOR F = 0…0` hoặc `= 1…1` ⇔ `r = F` hoặc `r = bù(F)` (với `bù(F) = F XOR 1…1`). Do đó với cùng một `F`, các hàng đều chính xác là các hàng **bằng `F` hoặc bằng bù của `F`**.

In [2]:
def complement(row):
    return [1 - v for v in row]


def apply_flips(matrix, flips):
    """Lật các cột j có flips[j] = 1."""
    return [[v ^ flips[j] for j, v in enumerate(row)] for row in matrix]


def count_equal_rows(matrix):
    return sum(1 for row in matrix if len(set(row)) == 1)


mat = [[0, 0, 0], [0, 0, 1], [1, 1, 0]]
print(apply_flips(mat, [0, 0, 1]), count_equal_rows(apply_flips(mat, [0, 0, 1])))
assert count_equal_rows(apply_flips(mat, [0, 0, 1])) == 2

[[0, 0, 1], [0, 0, 0], [1, 1, 1]] 2


### 3.3 Lớp tương đương {r, bù(r)}

Quan hệ "giống hệt hoặc bù nhau" là quan hệ tương đương trên các hàng (phản xạ, đối xứng; bắc cầu vì `bù(bù(r)) = r`). Nếu chọn `F = r` thì mọi hàng trong lớp của `r` đều đều; hàng ngoài lớp thì không. Do đó **đáp án = kích thước lớp lớn nhất**.

## 4. Ý tưởng nền tảng / trực giác

Chọn "hàng mẫu" `r` và lật cột sao cho `r` thành toàn `0`: lật đúng các cột mà `r` có `1`. Khi đó hàng nào **bằng `r`** cũng thành toàn `0`; hàng nào **bằng bù(r)** thành toàn `1`; hàng khác thì có cả `0` lẫn `1`. Vậy với mỗi hàng mẫu, số hàng đều = số hàng giống nó + số hàng bù của nó. Tối ưu: lấy mẫu tốt nhất. Để khỏi so sánh từng cặp, **chuẩn hoá** mỗi hàng về dạng đại diện chung của `{r, bù(r)}` rồi đếm.

## 5. Brute force — thử mọi tập cột lật

**Ý tưởng:** duyệt mọi `F ∈ {0,1}^n` (`2^n` tập), lật, đếm số hàng đều, lấy lớn nhất. Đúng theo định nghĩa, làm **oracle** (`n ≤ 12`).

In [3]:
def max_rows_brute(matrix):
    n = len(matrix[0])
    best = 0
    for flips in itertools.product((0, 1), repeat=n):
        best = max(best, count_equal_rows(apply_flips(matrix, flips)))
    return best


for mat, want in examples:
    assert max_rows_brute(mat) == want
print([max_rows_brute(m) for m, _ in examples])

[1, 2, 2]


In [4]:
# Code nộp LeetCode cho Brute force (đúng nhưng 2^n · m · n: chỉ chạy được khi n nhỏ)
class SolutionBrute:
    def maxEqualRowsAfterFlips(self, matrix: List[List[int]]) -> int:
        m, n = len(matrix), len(matrix[0])
        best = 0
        for mask in range(1 << n):
            cnt = 0
            for row in matrix:
                first = row[0] ^ (mask & 1)
                if all((row[j] ^ (mask >> j & 1)) == first for j in range(n)):
                    cnt += 1
            best = max(best, cnt)
        return best

**Độ phức tạp:** `O(2^n · m · n)` thời gian, `O(n)` bộ nhớ phụ.

**Nút thắt:** chỉ có `m ≤ 300` hàng nên chỉ có tối đa `m` mặt nạ `F` đáng thử (`F = hàng` nào đó), không phải `2^n`.

## 6. Bước đột phá

Mặt nạ tối ưu luôn có thể chọn là **một trong các hàng** (hoặc bù của nó). Với mặt nạ `F = r_i`, số hàng đều là số hàng bằng `r_i` hoặc `bù(r_i)`. Gộp mọi hàng cùng lớp `{r, bù(r)}` bằng cách **chuẩn hoá** và đếm: `O(m·n)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** so sánh từng cặp hàng, **Cách 3** chuẩn hoá bằng XOR với phần tử đầu rồi đếm (đáp án chính), **Cách 4** khoá `min(hàng, bù)`, **Cách 5** biểu diễn hàng bằng số nguyên bit.

### Cách 2 — Với mỗi hàng, đếm các hàng giống hoặc bù

**Giải thích:** cho mỗi hàng `r_i` làm "mẫu", đếm số hàng `r_j` thoả `r_j == r_i` hoặc `r_j == bù(r_i)`; lấy lớn nhất. `O(m² · n)`.

In [5]:
def max_rows_pairs(matrix):
    best = 0
    for r in matrix:
        c = complement(r)
        best = max(best, sum(1 for s in matrix if s == r or s == c))
    return best


for mat, want in examples:
    assert max_rows_pairs(mat) == want
print([max_rows_pairs(m) for m, _ in examples])

[1, 2, 2]


In [6]:
# Code nộp LeetCode cho Cách 2
class SolutionPairs:
    def maxEqualRowsAfterFlips(self, matrix: List[List[int]]) -> int:
        best = 0
        for r in matrix:
            c = [1 - v for v in r]
            cnt = 0
            for s in matrix:
                if s == r or s == c:
                    cnt += 1
            best = max(best, cnt)
        return best

**Độ phức tạp:** `O(m² · n)` thời gian (`300³ = 2.7·10^7` phép so sánh phần tử trong trường hợp xấu, nhưng `==` của list chạy bằng C nên nhanh), `O(n)` bộ nhớ.

### Cách 3 — Chuẩn hoá bằng XOR với phần tử đầu rồi đếm (đáp án chính)

**Ý tưởng:** biến mỗi hàng `r` thành `(v XOR r[0] for v in r)`: nếu `r[0] = 0` giữ nguyên, nếu `r[0] = 1` lật cả hàng (bù). Hai hàng cùng lớp `{r, bù(r)}` cho **cùng một** dạng chuẩn (luôn bắt đầu bằng `0`). Đếm bằng `Counter` các `tuple`, lấy tần suất lớn nhất.

In [7]:
def max_rows_normalize(matrix):
    cnt = Counter(tuple(v ^ row[0] for v in row) for row in matrix)
    return max(cnt.values())


for mat, want in examples:
    assert max_rows_normalize(mat) == want
print([max_rows_normalize(m) for m, _ in examples])

[1, 2, 2]


In [8]:
# Code nộp LeetCode cho Cách 3
class SolutionNormalize:
    def maxEqualRowsAfterFlips(self, matrix: List[List[int]]) -> int:
        cnt = Counter(tuple(v ^ row[0] for v in row) for row in matrix)
        return max(cnt.values())

**Độ phức tạp:** `O(m · n)` thời gian, `O(m · n)` bộ nhớ.

### Cách 4 — Khoá `min(hàng, bù)`

**Ý tưởng:** đại diện của lớp `{r, bù(r)}` là cái **nhỏ hơn theo thứ tự từ điển** của hai chuỗi. Dùng chuỗi `"0101…"` làm khoá để băm nhanh.

In [9]:
def max_rows_minkey(matrix):
    cnt = Counter()
    for row in matrix:
        a = "".join(map(str, row))
        b = "".join("1" if ch == "0" else "0" for ch in a)
        cnt[min(a, b)] += 1
    return max(cnt.values())


for mat, want in examples:
    assert max_rows_minkey(mat) == want
print([max_rows_minkey(m) for m, _ in examples])

[1, 2, 2]


In [10]:
# Code nộp LeetCode cho Cách 4
class SolutionMinKey:
    def maxEqualRowsAfterFlips(self, matrix: List[List[int]]) -> int:
        cnt = Counter()
        for row in matrix:
            a = "".join(map(str, row))
            b = a.translate(str.maketrans("01", "10"))
            cnt[min(a, b)] += 1
        return max(cnt.values())

**Độ phức tạp:** `O(m · n)` thời gian, `O(m · n)` bộ nhớ.

### Cách 5 — Số nguyên bit

**Ý tưởng:** coi hàng là một số nhị phân `x` (`n` bit), bù là `x ^ FULL` với `FULL = 2^n - 1`; khoá `min(x, x ^ FULL)`. So sánh số nguyên nhanh và gọn bộ nhớ.

In [11]:
def max_rows_bits(matrix):
    n = len(matrix[0])
    full = (1 << n) - 1
    cnt = Counter()
    for row in matrix:
        x = 0
        for v in row:
            x = x << 1 | v
        cnt[min(x, x ^ full)] += 1
    return max(cnt.values())


for mat, want in examples:
    assert max_rows_bits(mat) == want
print([max_rows_bits(m) for m, _ in examples])

[1, 2, 2]


In [12]:
# Code nộp LeetCode cho Cách 5
class SolutionBits:
    def maxEqualRowsAfterFlips(self, matrix: List[List[int]]) -> int:
        n = len(matrix[0])
        full = (1 << n) - 1
        cnt = Counter()
        for row in matrix:
            x = 0
            for v in row:
                x = x << 1 | v
            cnt[min(x, x ^ full)] += 1
        return max(cnt.values())

**Độ phức tạp:** `O(m · n)` thời gian (dựng số nguyên `n` bit), `O(m)` số nguyên `n` bit.

## 8. TRACE từng bước

Trace **Cách 3** với ví dụ 3: `matrix = [[0,0,0],[0,0,1],[1,1,0]]`.

| hàng | phần tử đầu | chuẩn hoá `v XOR r[0]` | ý nghĩa |
|---|---|---|---|
| `[0,0,0]` | 0 | `(0,0,0)` | giữ nguyên |
| `[0,0,1]` | 0 | `(0,0,1)` | giữ nguyên |
| `[1,1,0]` | 1 | `(0,0,1)` | lật cả hàng: là bù của `[0,0,1]` |

`Counter = {(0,0,0): 1, (0,0,1): 2}` ⇒ tần suất lớn nhất `2`. Hai hàng `[0,0,1]` và `[1,1,0]` là bù của nhau; lật cột `2` (cột cuối) biến chúng thành `[0,0,0]` và `[1,1,1]`: cả hai đều. Hàm trace in từng bước:

In [13]:
def trace_normalize(matrix):
    cnt = Counter()
    print("matrix =", matrix)
    for i, row in enumerate(matrix):
        key = tuple(v ^ row[0] for v in row)
        cnt[key] += 1
        print(f"  hàng {i} = {row}: phần tử đầu {row[0]} -> khoá {key} (đếm {cnt[key]})")
    best = max(cnt.values())
    print("  Counter =", dict(cnt), "-> đáp án", best)
    return best


assert trace_normalize([[0, 0, 0], [0, 0, 1], [1, 1, 0]]) == 2
assert trace_normalize([[0, 1], [1, 0]]) == 2

matrix = [[0, 0, 0], [0, 0, 1], [1, 1, 0]]
  hàng 0 = [0, 0, 0]: phần tử đầu 0 -> khoá (0, 0, 0) (đếm 1)
  hàng 1 = [0, 0, 1]: phần tử đầu 0 -> khoá (0, 0, 1) (đếm 1)
  hàng 2 = [1, 1, 0]: phần tử đầu 1 -> khoá (0, 0, 1) (đếm 2)
  Counter = {(0, 0, 0): 1, (0, 0, 1): 2} -> đáp án 2
matrix = [[0, 1], [1, 0]]
  hàng 0 = [0, 1]: phần tử đầu 0 -> khoá (0, 1) (đếm 1)
  hàng 1 = [1, 0]: phần tử đầu 1 -> khoá (0, 1) (đếm 2)
  Counter = {(0, 1): 2} -> đáp án 2


**Khôi phục tập cột cần lật:** lấy hàng đại diện của lớp lớn nhất `r` (hàng đầu tiên thuộc lớp đó) và lật đúng các cột mà `r` có `1` (khi đó `r` thành toàn `0`).

In [14]:
def best_flips(matrix):
    cnt = Counter(tuple(v ^ row[0] for v in row) for row in matrix)
    key = max(cnt, key=cnt.get)
    row = next(r for r in matrix if tuple(v ^ r[0] for v in r) == key)
    flips = list(row)                                       # lật các cột mà hàng mẫu có 1: hàng mẫu thành toàn 0
    return cnt[key], flips


c, flips = best_flips([[0, 0, 0], [0, 0, 1], [1, 1, 0]])
print(c, flips, apply_flips([[0, 0, 0], [0, 0, 1], [1, 1, 0]], flips))
assert c == 2 and count_equal_rows(apply_flips([[0, 0, 0], [0, 0, 1], [1, 1, 0]], flips)) == 2

2 [0, 0, 1] [[0, 0, 1], [0, 0, 0], [1, 1, 1]]


## 9. Code nộp LeetCode

Chọn **Cách 3** (chuẩn hoá bằng XOR với phần tử đầu). Giữ nguyên `class Solution`, `maxEqualRowsAfterFlips(self, matrix)`:

In [15]:
class Solution:
    def maxEqualRowsAfterFlips(self, matrix: List[List[int]]) -> int:
        cnt = Counter()
        for row in matrix:
            # chuẩn hoá: nếu phần tử đầu là 1 thì lật cả hàng; {r, bù(r)} có cùng khoá, luôn bắt đầu bằng 0
            cnt[tuple(v ^ row[0] for v in row)] += 1
        return max(cnt.values())


print([Solution().maxEqualRowsAfterFlips(m) for m, _ in examples])
assert all(Solution().maxEqualRowsAfterFlips(m) == v for m, v in examples)

[1, 2, 2]


## 10. Vì sao đúng

**(a) Hàng đều sau lật.** Với tập cột lật `F`, hàng `r` thành `r XOR F` (mục 3.1). Nó đều ⇔ tất cả `r[j] XOR F[j]` bằng nhau ⇔ `r XOR F ∈ {0…0, 1…1}` ⇔ `r ∈ {F, bù(F)}`.

**(b) Số hàng đều với mặt nạ `F`.** `h(F) = #{i : r_i = F} + #{i : r_i = bù(F)}` (với `n ≥ 1`, `F ≠ bù(F)` nên hai tập rời nhau). Đáp án `= max_F h(F)`.

**(c) Chỉ cần `F` là một hàng.** Nếu `h(F) > 0` thì có hàng `r_i ∈ {F, bù(F)}`; `h(r_i) = h(F)` vì `{r_i, bù(r_i)} = {F, bù(F)}`. Nếu mọi `h(F) = 0` thì không có hàng nào đều, nhưng với `m ≥ 1`, lấy `F = r_1` cho `h ≥ 1`. Vậy `max_F h(F) = max_i h(r_i)`.

**(d) Chuẩn hoá.** Với `r[0] = 0`, `norm(r) = r`; với `r[0] = 1`, `norm(r) = bù(r)`. Hai hàng có cùng `norm` ⇔ cùng lớp `{r, bù(r)}`: nếu `r[0] = s[0]` thì `norm` bằng nhau ⇔ `r = s`; nếu `r[0] ≠ s[0]` thì `norm` bằng nhau ⇔ `r = bù(s)`. Do đó số hàng có cùng `norm` với `r_i` đúng bằng `h(r_i)`, và đáp án `= max` tần suất.

**(e) Các cách còn lại.** Cách 2 tính `h(r_i)` trực tiếp. Cách 4 và 5 chọn đại diện khác (`min` từ điển / `min` số nguyên) của cùng lớp. Cách 1 là định nghĩa.

**(f) Kiểm chứng bằng máy.** Mọi cách khớp vét cạn trên mọi ma trận nhị phân `m × n` với `m·n ≤ 12` (mục 12) và các ma trận ngẫu nhiên.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn | `O(2^n · m · n)` | `O(n)` | oracle `n ≤ 12` |
| 2. So sánh cặp | `O(m² · n)` | `O(n)` | không cần băm |
| 3. Chuẩn hoá XOR | `O(m · n)` | `O(m · n)` | **đáp án chính** |
| 4. Khoá `min(a, bù)` chuỗi | `O(m · n)` | `O(m · n)` | dễ đọc |
| 5. Số nguyên bit | `O(m · n)` | `O(m)` số `n` bit | gọn bộ nhớ |

Đo thực tế (`m = n = 300`):

In [16]:
def bench(fn, arg, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(1072)
rand = [[rng.randint(0, 1) for _ in range(300)] for _ in range(300)]
pattern = [rng.randint(0, 1) for _ in range(300)]
struct = [pattern if i % 2 == 0 else complement(pattern) for i in range(300)]
for label, mat in [("ngẫu nhiên", rand), ("một lớp duy nhất", struct)]:
    print(label)
    for name, fn in [("pairs", max_rows_pairs), ("normalize", max_rows_normalize), ("minkey", max_rows_minkey), ("bits", max_rows_bits)]:
        print(f"  {name:10s}: {bench(fn, mat) * 1000:8.2f} ms -> {fn(mat)}")

ngẫu nhiên


  pairs     :    12.04 ms -> 1


  normalize :     6.58 ms -> 1


  minkey    :    15.58 ms -> 1
  bits      :     9.97 ms -> 1
một lớp duy nhất


  pairs     :    44.26 ms -> 300


  normalize :     3.62 ms -> 300
  minkey    :     8.71 ms -> 300
  bits      :     5.66 ms -> 300


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi ma trận nhỏ, (E) tính chất / ngẫu nhiên (seed cố định), (F) giới hạn.

### A. Ví dụ của đề

In [17]:
IMPLS = {
    "pairs": max_rows_pairs,
    "normalize": max_rows_normalize,
    "minkey": max_rows_minkey,
    "bits": max_rows_bits,
    "Solution": lambda m: Solution().maxEqualRowsAfterFlips(m),
    "SolutionPairs": lambda m: SolutionPairs().maxEqualRowsAfterFlips(m),
    "SolutionNormalize": lambda m: SolutionNormalize().maxEqualRowsAfterFlips(m),
    "SolutionMinKey": lambda m: SolutionMinKey().maxEqualRowsAfterFlips(m),
    "SolutionBits": lambda m: SolutionBits().maxEqualRowsAfterFlips(m),
}
BRUTE = {"brute": max_rows_brute, "SolutionBrute": lambda m: SolutionBrute().maxEqualRowsAfterFlips(m)}


def check(matrix, want=None, brute=False):
    if want is None:
        want = max_rows_brute(matrix) if brute else Solution().maxEqualRowsAfterFlips(matrix)
    impls = dict(IMPLS)
    if brute:
        impls.update(BRUTE)
    for name, fn in impls.items():
        got = fn([r[:] for r in matrix])
        assert got == want, (name, matrix, got, want)
    return want


for mat, want in examples:
    check(mat, want, brute=True)
print("ví dụ đề: mọi cài đặt đúng;", len(IMPLS), "cách nhanh +", len(BRUTE), "vét cạn")

ví dụ đề: mọi cài đặt đúng; 9 cách nhanh + 2 vét cạn


### B. Edge case

- Ma trận `1 × 1`: luôn `1` (một ô là hàng đều).
- Một hàng bất kỳ: luôn `1`.
- Một cột: mọi hàng đều có một phần tử nên đều ⇒ đáp án `m`.
- Mọi hàng giống nhau; mọi hàng là `r` hoặc `bù(r)`.
- Mọi hàng đôi một khác lớp (đáp án `1`).
- Toàn `0` / toàn `1`.
- Ma trận có hàng đều sẵn nhưng không cùng lớp (`[0,0]` và `[1,1]` cùng lớp; `[0,0,0]` và `[0,1,0]` không).

In [18]:
edge = [
    ([[0]], 1), ([[1]], 1), ([[0, 1, 1, 0]], 1), ([[0], [1], [1], [0]], 4), ([[1], [0]], 2),
    ([[0, 0], [0, 0], [0, 0]], 3), ([[1, 1, 1], [1, 1, 1]], 2), ([[0, 1, 0], [0, 1, 0], [0, 1, 0]], 3),
    ([[0, 1, 0], [1, 0, 1], [0, 1, 0], [1, 0, 1], [0, 0, 0]], 4),
    ([[0, 0, 0], [0, 1, 0]], 1),
    ([[0, 0], [1, 1], [0, 1]], 2),
    ([[0, 1, 1], [1, 0, 0], [1, 0, 1], [0, 1, 0]], 2),
]
for mat, want in edge:
    got = check(mat, want, brute=True)
    print(f"  {mat} -> {got}")
print(len(edge), "edge case: mọi cài đặt (kể cả vét cạn) khớp")

  [[0]] -> 1
  [[1]] -> 1
  [[0, 1, 1, 0]] -> 1
  [[0], [1], [1], [0]] -> 4
  [[1], [0]] -> 2
  [[0, 0], [0, 0], [0, 0]] -> 3
  [[1, 1, 1], [1, 1, 1]] -> 2
  [[0, 1, 0], [0, 1, 0], [0, 1, 0]] -> 3
  [[0, 1, 0], [1, 0, 1], [0, 1, 0], [1, 0, 1], [0, 0, 0]] -> 4
  [[0, 0, 0], [0, 1, 0]] -> 1
  [[0, 0], [1, 1], [0, 1]] -> 2
  [[0, 1, 1], [1, 0, 0], [1, 0, 1], [0, 1, 0]] -> 2
12 edge case: mọi cài đặt (kể cả vét cạn) khớp


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_identical_only(matrix):                         # chỉ đếm hàng giống hệt, bỏ qua bù
    return max(Counter(map(tuple, matrix)).values())


def bug_count_ones(matrix):                             # nhóm theo số lượng số 1
    return max(Counter(sum(r) for r in matrix).values())


def bug_already_equal(matrix):                          # chỉ đếm hàng đã đều sẵn
    return max(1, count_equal_rows(matrix))


def bug_reverse_not_complement(matrix):                 # nhầm bù với đảo ngược
    cnt = Counter()
    for r in matrix:
        cnt[min(tuple(r), tuple(r[::-1]))] += 1
    return max(cnt.values())


def bug_flip_rows(matrix):                              # lật HÀNG thay vì lật cột: chỉ cần 1 phần tử khác nhau là sai
    return max(1, count_equal_rows(matrix))


def bug_first_row_only(matrix):                         # chỉ lấy hàng đầu làm mẫu
    r = matrix[0]
    c = complement(r)
    return sum(1 for s in matrix if s == r or s == c)


BUGS = [bug_identical_only, bug_count_ones, bug_already_equal, bug_reverse_not_complement, bug_first_row_only]
rng = random.Random(10720)
pool = []
for _ in range(2000):
    m, n = rng.randint(1, 5), rng.randint(1, 5)
    pool.append([[rng.randint(0, 1) for _ in range(n)] for _ in range(m)])
for bug in BUGS:
    found = next((mat for mat in pool if bug(mat) != max_rows_brute(mat)), None)
    assert found, bug.__name__
    print(f"{bug.__name__:26s} sai trên {found}: ra {bug(found)}, đúng {max_rows_brute(found)}")

bug_identical_only         sai trên [[1, 0, 0, 1], [1, 1, 1, 0], [1, 0, 0, 1], [0, 0, 0, 1], [0, 1, 1, 0]]: ra 2, đúng 3
bug_count_ones             sai trên [[1, 1], [0, 1], [0, 0]]: ra 1, đúng 2
bug_already_equal          sai trên [[1, 0, 0, 1], [1, 1, 1, 0], [1, 0, 0, 1], [0, 0, 0, 1], [0, 1, 1, 0]]: ra 1, đúng 3
bug_reverse_not_complement sai trên [[1, 0, 0, 1], [1, 1, 1, 0], [1, 0, 0, 1], [0, 0, 0, 1], [0, 1, 1, 0]]: ra 2, đúng 3
bug_first_row_only         sai trên [[0, 1, 0], [1, 0, 0], [1, 0, 0]]: ra 1, đúng 2


### D. Vét cạn mọi ma trận nhỏ

Mọi ma trận nhị phân `m × n` với `1 ≤ m, n ≤ 4` và `m · n ≤ 12` (tức `(m, n)` trừ `(4, 4)`): tổng cộng `Σ 2^(m·n)` ma trận. So mọi cài đặt với vét cạn.

In [20]:
t0 = time.perf_counter()
cnt = 0
for m in range(1, 5):
    for n in range(1, 5):
        if m * n > 12:
            continue
        for bits in range(1 << (m * n)):
            mat = [[(bits >> (i * n + j)) & 1 for j in range(n)] for i in range(m)]
            want = max_rows_brute(mat)
            for name, fn in IMPLS.items():
                assert fn(mat) == want, (name, mat)
            cnt += 1
print(cnt, "ma trận (m·n <= 12):", len(IMPLS), "cài đặt khớp vét cạn;", round(time.perf_counter() - t0, 1), "s")

9418 ma trận (m·n <= 12): 9 cài đặt khớp vét cạn; 0.5 s


### E. Tính chất / ngẫu nhiên (seed cố định)

(1) `n ≤ 10`: mọi cách khớp vét cạn; (2) `300 × 300`: các cách khớp nhau; (3) **hoán vị hàng** không đổi; (4) **hoán vị cột** không đổi; (5) **lật một cột bất kỳ** không đổi đáp án; (6) **bù cả ma trận** không đổi; (7) **thêm một hàng** không làm giảm (và tăng tối đa `1`); (8) **chuỗi gài**: chèn `k` hàng `p` hoặc bù của `p` ⇒ đáp án `≥ k`; (9) `1 ≤ ans ≤ m`.

In [21]:
rng = random.Random(20240172)
t0 = time.perf_counter()
for it in range(500):
    m, n = rng.randint(1, 8), rng.randint(1, 10)
    mat = [[rng.randint(0, 1) for _ in range(n)] for _ in range(m)]
    check(mat, brute=True)                                                            # (1)
print("(1) 500 ma trận n <= 10 khớp vét cạn:", round(time.perf_counter() - t0, 1), "s")

t0 = time.perf_counter()
for it in range(150):
    m, n = rng.randint(1, 300), rng.randint(1, 300)
    p = [rng.randint(0, 1) for _ in range(n)]
    k = rng.randint(0, m)
    mat = [(p if rng.random() < 0.5 else complement(p))[:] for _ in range(k)] + [[rng.randint(0, 1) for _ in range(n)] for _ in range(m - k)]
    rng.shuffle(mat)
    ans = check(mat)                                                                  # (2)
    if n >= 12:
        assert ans >= k                                                               # (8)
    assert 1 <= ans <= m                                                              # (9)
    m2 = [r[:] for r in mat]
    rng.shuffle(m2)
    assert Solution().maxEqualRowsAfterFlips(m2) == ans                               # (3)
    perm = list(range(n))
    rng.shuffle(perm)
    assert Solution().maxEqualRowsAfterFlips([[r[j] for j in perm] for r in mat]) == ans   # (4)
    col = rng.randrange(n)
    assert Solution().maxEqualRowsAfterFlips([[v ^ (j == col) for j, v in enumerate(r)] for r in mat]) == ans   # (5)
    assert Solution().maxEqualRowsAfterFlips([complement(r) for r in mat]) == ans     # (6)
    newrow = [rng.randint(0, 1) for _ in range(n)]
    ans2 = Solution().maxEqualRowsAfterFlips(mat + [newrow])
    assert ans <= ans2 <= ans + 1                                                     # (7)
print("(2)-(9) 150 ma trận tới 300 × 300:", round(time.perf_counter() - t0, 1), "s")

(1) 500 ma trận n <= 10 khớp vét cạn: 1.0 s


(2)-(9) 150 ma trận tới 300 × 300: 5.1 s


### F. Giới hạn

`300 × 300`: toàn `0`; toàn một lớp `{p, bù(p)}` (đáp án `300`); mọi hàng đôi một khác lớp (đáp án `1` nếu thật sự khác lớp); ngẫu nhiên; `1 × 300`; `300 × 1`. Mọi cách nhanh khớp nhau.

In [22]:
t0 = time.perf_counter()
rg = random.Random(3)
p = [rg.randint(0, 1) for _ in range(300)]
# 300 hàng đôi một khác lớp: dùng hàng "một-hot" e_i (i = 0..299) đã chuẩn hoá khác nhau
one_hot = [[1 if j == i else 0 for j in range(300)] for i in range(300)]
cases = {
    "toàn 0": ([[0] * 300 for _ in range(300)], 300),
    "một lớp": ([p[:] if i % 3 else complement(p) for i in range(300)], 300),
    "đôi một khác lớp": (one_hot, 1),
    "ngẫu nhiên": ([[rg.randint(0, 1) for _ in range(300)] for _ in range(300)], None),
    "1 × 300": ([[rg.randint(0, 1) for _ in range(300)]], 1),
    "300 × 1": ([[rg.randint(0, 1)] for _ in range(300)], 300),
}
for name, (mat, want) in cases.items():
    ans = check(mat, want)
    print(f"  {name:18s}: {len(mat)} × {len(mat[0])} -> {ans}")
print("giới hạn:", round(time.perf_counter() - t0, 1), "s")

  toàn 0            : 300 × 300 -> 300


  một lớp           : 300 × 300 -> 300
  đôi một khác lớp  : 300 × 300 -> 1


  ngẫu nhiên        : 300 × 300 -> 1
  1 × 300           : 1 × 300 -> 1
  300 × 1           : 300 × 1 -> 300
giới hạn: 0.4 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ đếm hàng giống hệt

In [23]:
mat = [[0, 1], [1, 0]]
print("bug:", bug_identical_only(mat), "| đúng:", Solution().maxEqualRowsAfterFlips(mat))
assert bug_identical_only(mat) == 1 and Solution().maxEqualRowsAfterFlips(mat) == 2

bug: 1 | đúng: 2


**Vì sao sai:** hai hàng **bù nhau** (như `[0,1]` và `[1,0]`) cũng cùng đều sau cùng một cách lật cột (lật cột đầu: `[1,1]` và `[0,0]`). Phải gộp `r` và `bù(r)` cùng lớp.

### Lỗi 2: nhóm theo số lượng số `1`

In [24]:
bad = next(mat for mat in pool if bug_count_ones(mat) != max_rows_brute(mat))
print("phản ví dụ:", bad, "| bug:", bug_count_ones(bad), "| đúng:", max_rows_brute(bad))
assert bug_count_ones(bad) != max_rows_brute(bad)

phản ví dụ: [[1, 1], [0, 1], [0, 0]] | bug: 1 | đúng: 2


**Vì sao sai:** hai hàng có cùng số `1` nhưng khác vị trí (`[1,0,0]` và `[0,1,0]`) cần các cột lật **khác nhau**; không cùng đều.

### Lỗi 3: chỉ đếm hàng đã đều sẵn

In [25]:
mat = [[0, 1], [0, 1], [1, 0]]
print("bug:", bug_already_equal(mat), "| đúng:", Solution().maxEqualRowsAfterFlips(mat))
assert bug_already_equal(mat) == 1 and Solution().maxEqualRowsAfterFlips(mat) == 3

bug: 1 | đúng: 3


**Vì sao sai:** ta **được lật cột** nên hàng không đều sẵn vẫn có thể thành đều (`[0,1]` đều khi lật cột `1`). Không chỉ đếm hàng đang đều.

### Lỗi 4: nhầm "bù" với "đảo ngược"

In [26]:
bad = next(mat for mat in pool if bug_reverse_not_complement(mat) != max_rows_brute(mat))
print("phản ví dụ:", bad, "| bug:", bug_reverse_not_complement(bad), "| đúng:", max_rows_brute(bad))
assert bug_reverse_not_complement(bad) != max_rows_brute(bad)

phản ví dụ: [[1, 0, 0, 1], [1, 1, 1, 0], [1, 0, 0, 1], [0, 0, 0, 1], [0, 1, 1, 0]] | bug: 2 | đúng: 3


**Vì sao sai:** lật cột đổi `0 ↔ 1` **tại chỗ**, không đổi thứ tự. Bù là `1 - v` từng phần tử, **không phải** đảo ngược thứ tự phần tử.

### Lỗi 5: chỉ lấy hàng đầu làm mẫu

In [27]:
bad = next(mat for mat in pool if bug_first_row_only(mat) != max_rows_brute(mat))
print("phản ví dụ:", bad, "| bug:", bug_first_row_only(bad), "| đúng:", max_rows_brute(bad))
assert bug_first_row_only(bad) < max_rows_brute(bad)

phản ví dụ: [[0, 1, 0], [1, 0, 0], [1, 0, 0]] | bug: 1 | đúng: 2


**Vì sao sai:** lớp lớn nhất có thể không chứa hàng đầu. Phải thử mọi hàng làm mẫu (hoặc đếm mọi lớp).

### Lỗi 6 (bonus): lật hàng thay vì lật cột

In [28]:
mat = [[0, 1], [1, 0]]
print("lật HÀNG thì không giải quyết được [0,1] (hàng vẫn không đều):", count_equal_rows(mat), "| lật cột 0:", count_equal_rows(apply_flips(mat, [1, 0])))
assert count_equal_rows(mat) == 0 and count_equal_rows(apply_flips(mat, [1, 0])) == 2

lật HÀNG thì không giải quyết được [0,1] (hàng vẫn không đều): 0 | lật cột 0: 2


**Vì sao sai:** thao tác của đề là lật **cột** (ảnh hưởng mọi hàng cùng lúc), nên các hàng liên đới với nhau; không thể xử lý từng hàng độc lập.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "thao tác áp dụng **đồng loạt** lên mọi hàng (lật cột), tối đa số hàng đạt tính chất" → các hàng "tương đương" nếu chúng **khác nhau đúng một phép biến đổi chung** → chuẩn hoá về đại diện và **đếm bằng bảng băm**.

**Mẹo:**

1. Lật cột = XOR cùng mặt nạ lên mọi hàng; hàng đều ⇔ hàng `= F` hoặc `bù(F)`.
2. Chuẩn hoá bằng `v ^ row[0]` (hàng luôn bắt đầu bằng `0`).
3. Dùng `tuple` (hoặc chuỗi / số nguyên bit) làm khoá băm; `list` không băm được.
4. Đáp án là `max` tần suất khoá.
5. Oracle: vét cạn `2^n` mặt nạ cho `n ≤ 12`.
6. Kiểm tra bất biến: hoán vị hàng/cột, lật một cột, bù cả ma trận.

**Họ bài cùng pattern:** Number of Distinct Islands (694), Group Anagrams (49), Max Points on a Line (149), Count Number of Nice Subarrays (1248), Minimum Number of K Consecutive Bit Flips (995), Remove Colored Pieces (2038).

## 15. Biến thể

1. **Số lần lật tối thiểu** để đạt số hàng đều lớn nhất.
2. **Khôi phục tập cột cần lật** (đã làm ở mục 8).
3. **Đếm số lớp** đạt tần suất lớn nhất (có bao nhiêu cách chọn lớp tối ưu).
4. **Cho phép lật cả hàng và cột:** mọi hàng đều có thể đều (lật hàng không giúp: hàng đều ⇔ hàng đều sau lật). Bài khác: *tối thiểu số lần lật hàng/cột để mọi ô bằng nhau* — ở đây chỉ kiểm chứng biến thể 1 và 3.

Cài đặt và đối chiếu vét cạn:

In [29]:
def min_flips_for_max(matrix):
    """Số cột lật ít nhất để đạt số hàng đều lớn nhất: với hàng mẫu r, lật các cột r có 1 (làm r thành 0...0) hoặc các cột r có 0 (thành 1...1)."""
    n = len(matrix[0])
    cnt = Counter(tuple(v ^ row[0] for v in row) for row in matrix)
    top = max(cnt.values())
    best = None
    for key, c in cnt.items():
        if c == top:
            ones = sum(key)                              # khoá bắt đầu bằng 0; lật các cột có 1 hoặc các cột có 0
            cost = min(ones, n - ones)
            best = cost if best is None else min(best, cost)
    return top, best


def count_best_classes(matrix):
    cnt = Counter(tuple(v ^ row[0] for v in row) for row in matrix)
    top = max(cnt.values())
    return sum(1 for c in cnt.values() if c == top)


def brute_variants(matrix):
    n = len(matrix[0])
    best, min_flips, masks = -1, None, []
    for flips in itertools.product((0, 1), repeat=n):
        c = count_equal_rows(apply_flips(matrix, flips))
        if c > best:
            best, min_flips, masks = c, sum(flips), [flips]
        elif c == best:
            min_flips = min(min_flips, sum(flips))
            masks.append(flips)
    return best, min_flips


rng = random.Random(15)
for _ in range(500):
    m, n = rng.randint(1, 6), rng.randint(1, 7)
    mat = [[rng.randint(0, 1) for _ in range(n)] for _ in range(m)]
    best, mf = brute_variants(mat)
    assert min_flips_for_max(mat) == (best, mf), mat
    # số lớp tối ưu: đếm các lớp khác nhau đạt `best` hàng đều (kiểm bằng định nghĩa lớp)
    classes = {min(tuple(r), tuple(complement(r))) for r in mat}
    sizes = [sum(1 for r in mat if tuple(r) in (c, tuple(complement(list(c))))) for c in classes]
    assert count_best_classes(mat) == sum(1 for s in sizes if s == best)
print("số lần lật tối thiểu / đếm số lớp tối ưu: khớp vét cạn")

số lần lật tối thiểu / đếm số lớp tối ưu: khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử `2^n` mặt nạ | `O(2^n · m n)` | `O(n)` | oracle |
| 2. So sánh cặp | mẫu = mỗi hàng | `O(m² n)` | `O(n)` | không cần băm |
| 3. Chuẩn hoá XOR | `v ^ row[0]` | `O(m n)` | `O(m n)` | **đáp án chính** |
| 4. `min(a, bù)` chuỗi | đại diện từ điển | `O(m n)` | `O(m n)` | dễ đọc |
| 5. Số nguyên bit | `min(x, x ^ full)` | `O(m n)` | `O(m)` số lớn | gọn |

Mọi cách khớp vét cạn trên toàn bộ ma trận nhị phân nhỏ (`m·n ≤ 12`) và `500` ma trận ngẫu nhiên `n ≤ 10`; khớp nhau trên `300 × 300`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [30]:
print(Solution().maxEqualRowsAfterFlips([[1, 0], [0, 1], [1, 0]]), Solution().maxEqualRowsAfterFlips([[0, 0, 1], [1, 1, 0], [0, 1, 0]]), Solution().maxEqualRowsAfterFlips([[1]]), Solution().maxEqualRowsAfterFlips([[0, 1, 0], [1, 0, 1], [1, 1, 1], [0, 0, 0]]))

3 2 1 2


### Bài 2 (tính tay)

`matrix = [[0,1,1],[1,0,0],[1,0,1],[0,1,0]]`. Chuẩn hoá từng hàng bằng `v XOR r[0]`, lập `Counter`, rồi cho biết đáp án và tập cột cần lật. Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện chuẩn hoá + đếm:

In [31]:
def max_rows_todo(matrix):
    cnt = {}
    for row in matrix:
        key = None      # TODO 1: tuple(v ^ row[0] for v in row)
        cnt[key] = cnt.get(key, 0) + 1
    return 0            # TODO 2: tần suất lớn nhất


# Khi làm xong, bỏ comment để kiểm tra:
# assert all(max_rows_todo(m) == v for m, v in examples)

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **khoá số nguyên bit** `min(x, x ^ full)`:

In [32]:
class MySolution:
    def maxEqualRowsAfterFlips(self, matrix):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().maxEqualRowsAfterFlips(m) == v for m, v in examples)

### Bài 5 (mini challenge): số lần lật cột tối thiểu

Viết `min_flips(matrix)` trả về số cột lật **ít nhất** để đạt số hàng đều lớn nhất. Gợi ý: với lớp có khoá chuẩn hoá `key` (bắt đầu bằng `0`), lật các cột có `1` hoặc các cột có `0`: chọn `min(số số 1, n − số số 1)`; lấy nhỏ nhất trên các lớp đạt tần suất lớn nhất.

In [33]:
def min_flips(matrix):
    pass            # TODO


# Khi làm xong:
# assert min_flips([[0, 0, 0], [0, 0, 1], [1, 1, 0]]) == 1
# assert min_flips([[0, 1], [1, 0]]) == 1

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: chuẩn hoá mỗi hàng về dạng bắt đầu bằng `0`.
- Mức 2: `[[1,0],[0,1],[1,0]]` → `(0,1), (0,1), (0,1)`; `[[1]]` có một hàng.
- Mức 3: `3 2 1 2`.

**Bài 2**

- Mức 1: `[0,1,1] → (0,1,1)`; `[1,0,0] → (0,1,1)`.
- Mức 2: `[1,0,1] → (0,1,0)`; `[0,1,0] → (0,1,0)`.
- Mức 3: `Counter = {(0,1,1): 2, (0,1,0): 2}` ⇒ đáp án `2`; lật các cột mà hàng mẫu `[0,1,1]` có `1`: cột `1` và `2`.

**Bài 3**

- Mức 1: `key = tuple(v ^ row[0] for v in row)`.
- Mức 2: `cnt[key] = cnt.get(key, 0) + 1` đã có sẵn.
- Mức 3: `return max(cnt.values())`.

**Bài 4**

- Mức 1: `full = (1 << n) - 1`.
- Mức 2: dựng `x` từ hàng: `x = x << 1 | v`.
- Mức 3: `cnt[min(x, x ^ full)] += 1`; trả `max(cnt.values())`.

**Bài 5**

- Mức 1: lấy `cnt` các khoá chuẩn hoá và `top = max(cnt.values())`.
- Mức 2: với khoá `key` (bắt đầu bằng `0`) có `sum(key)` số `1`.
- Mức 3: `min(sum(key), n - sum(key))`; lấy min trên các khoá có `cnt = top`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [34]:
vals = (Solution().maxEqualRowsAfterFlips([[1, 0], [0, 1], [1, 0]]), Solution().maxEqualRowsAfterFlips([[0, 0, 1], [1, 1, 0], [0, 1, 0]]), Solution().maxEqualRowsAfterFlips([[1]]), Solution().maxEqualRowsAfterFlips([[0, 1, 0], [1, 0, 1], [1, 1, 1], [0, 0, 0]]))
print(vals)
assert vals == (3, 2, 1, 2)
for m_, v in zip(([[1, 0], [0, 1], [1, 0]], [[0, 0, 1], [1, 1, 0], [0, 1, 0]], [[1]], [[0, 1, 0], [1, 0, 1], [1, 1, 1], [0, 0, 0]]), vals):
    assert max_rows_brute(m_) == v
print("Bài 1 OK")

(3, 2, 1, 2)
Bài 1 OK


### Bài 2

In [35]:
m2 = [[0, 1, 1], [1, 0, 0], [1, 0, 1], [0, 1, 0]]
assert trace_normalize(m2) == 2 == max_rows_brute(m2)
c, flips = best_flips(m2)
print(c, flips)
assert c == 2 and flips == [0, 1, 1] and count_equal_rows(apply_flips(m2, flips)) >= 2
print("Bài 2 OK")

matrix = [[0, 1, 1], [1, 0, 0], [1, 0, 1], [0, 1, 0]]
  hàng 0 = [0, 1, 1]: phần tử đầu 0 -> khoá (0, 1, 1) (đếm 1)
  hàng 1 = [1, 0, 0]: phần tử đầu 1 -> khoá (0, 1, 1) (đếm 2)
  hàng 2 = [1, 0, 1]: phần tử đầu 1 -> khoá (0, 1, 0) (đếm 1)
  hàng 3 = [0, 1, 0]: phần tử đầu 0 -> khoá (0, 1, 0) (đếm 2)
  Counter = {(0, 1, 1): 2, (0, 1, 0): 2} -> đáp án 2
2 [0, 1, 1]
Bài 2 OK


### Bài 3

In [36]:
def max_rows_answer(matrix):
    cnt = {}
    for row in matrix:
        key = tuple(v ^ row[0] for v in row)
        cnt[key] = cnt.get(key, 0) + 1
    return max(cnt.values())


assert all(max_rows_answer(m) == v for m, v in examples)
for mat in pool[:1000]:
    assert max_rows_answer(mat) == max_rows_brute(mat)
print("Bài 3 OK")

Bài 3 OK

### Bài 4

In [37]:
class MySolutionAnswer:
    def maxEqualRowsAfterFlips(self, matrix):
        n = len(matrix[0])
        full = (1 << n) - 1
        cnt = {}
        for row in matrix:
            x = 0
            for v in row:
                x = x << 1 | v
            k = min(x, x ^ full)
            cnt[k] = cnt.get(k, 0) + 1
        return max(cnt.values())


assert all(MySolutionAnswer().maxEqualRowsAfterFlips(m) == v for m, v in examples)
for mat in pool[:1000]:
    assert MySolutionAnswer().maxEqualRowsAfterFlips(mat) == max_rows_brute(mat)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def min_flips(matrix):
    return min_flips_for_max(matrix)[1]


assert min_flips([[0, 0, 0], [0, 0, 1], [1, 1, 0]]) == 1
assert min_flips([[0, 1], [1, 0]]) == 1
for mat in pool[:1000]:
    assert min_flips(mat) == brute_variants(mat)[1]
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (thao tác đồng loạt lên mọi hàng, đếm số hàng "tương đương"):**

```python
class Solution:
    def maxEqualRowsAfterFlips(self, matrix: List[List[int]]) -> int:
        cnt = Counter()
        for row in matrix:
            cnt[tuple(v ^ row[0] for v in row)] += 1
        return max(cnt.values())
```

**Checklist trước khi submit:**

- [ ] Hiểu lật cột = XOR cùng một mặt nạ lên mọi hàng?
- [ ] Gộp cả hàng **giống** và hàng **bù** (chuẩn hoá về cùng đại diện)?
- [ ] Khoá băm là `tuple`/chuỗi/số nguyên (không dùng `list`)?
- [ ] Đáp án là `max` tần suất (luôn `≥ 1` vì `m ≥ 1`)?
- [ ] Đã thử `[[0,1],[1,0]] → 2`, `[[0,1],[1,1]] → 1`, một cột, một hàng?
- [ ] Đúng tên `class Solution`, `maxEqualRowsAfterFlips(self, matrix)`, không in debug.